# 🚑 Ambulance Dispatch — GRPO Training with HF TRL + Unsloth

This notebook trains a small LLM to dispatch ambulances using GRPO (Group Relative Policy Optimization).

**Environment**: City-scale ambulance dispatch (OpenEnv — RFC 001–005 compliant)  
**Model**: `unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit` (free T4 compatible, ~15 GB VRAM)  
**Training**: GRPO via HuggingFace TRL  
**Reward**: 9-component RFC 004 Rubric  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CSNEHA20/Meta_PyTorch_OpenEnv_Hackathon/blob/main/notebooks/Ambulance_GRPO_Training.ipynb)

> **Runtime**: Free Colab T4 (15 GB). Training 100 steps takes ~15–20 min.

In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# Run this first. Takes ~3-5 minutes.
# ============================================================
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# Core ML stack
pip("torch", "torchvision")
pip("transformers>=4.40", "datasets", "accelerate>=0.26")
pip("trl>=0.12.0")
pip("peft>=0.10.0")

# Environment dependencies
pip("networkx>=3.2", "numpy>=1.26", "pydantic>=2.0", "fastapi", "uvicorn")

# Unsloth for 2x faster 4-bit GRPO training
try:
    pip("unsloth[colab-new]@git+https://github.com/unslothai/unsloth.git")
    USE_UNSLOTH = True
    print("✅ Unsloth installed — 4-bit quantised GRPO enabled (2x faster)")
except Exception as e:
    print(f"⚠️  Unsloth failed ({e}), falling back to standard transformers")
    USE_UNSLOTH = False

# Plotting
pip("matplotlib", "seaborn")

print(f"\n✅ All dependencies installed. USE_UNSLOTH={USE_UNSLOTH}")

: 

In [ ]:
# ============================================================
# CELL 2: Clone Repository & Install Dependencies
# IMPORTANT: After this cell runs, you may need to restart the runtime
# (Runtime → Restart runtime) if you see openenv import errors in Cell 3
# ============================================================
import os, subprocess, sys

REPO_URL = "https://github.com/CSNEHA20/Meta_PyTorch_OpenEnv_Hackathon.git"
REPO_DIR = "Ambulance-OpenENV"

if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
    print(f"✅ Cloned {REPO_URL}")
else:
    # Pull latest changes
    subprocess.check_call(["git", "-C", REPO_DIR, "pull"])
    print(f"✅ Pulled latest from {REPO_URL}")

# Set working directory
if os.path.basename(os.getcwd()) != REPO_DIR:
    os.chdir(REPO_DIR)

# Add to Python path so env/ imports work
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"📁 Working directory: {os.getcwd()}")

# Install openenv (needed for env/models.py imports)
print("📦 Installing openenv...")
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openenv"])
    print("✅ openenv installed from PyPI")
except Exception as e1:
    print(f"⚠️ PyPI install failed: {e1}")
    # Try from requirements.txt if openenv is pinned there
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
        print("✅ requirements.txt installed")
    except Exception as e2:
        print(f"⚠️ requirements.txt failed: {e2}")

print("\n" + "="*60)
print("⚠️  IMPORTANT: If Cell 3 fails with 'No module named openenv'")
print("    Run: Runtime → Restart runtime → Run all")
print("="*60)

In [ ]:
# ============================================================
# CELL 3: Verify Environment
# Must print "✅ Env works!" before proceeding
# If you see "No module named 'openenv'", restart the runtime:
# Runtime → Restart runtime → then run all cells again
# ============================================================
import sys
import json

# Try to import with helpful error message
try:
    from env.environment import AmbulanceEnvironment
    from env.models import ActionModel, AmbulanceState, Severity
except ModuleNotFoundError as e:
    if "openenv" in str(e):
        print("❌ ERROR: 'openenv' package not found!")
        print("\n👉 FIX: Run 'Runtime → Restart runtime' in the menu above")
        print("   Then: Runtime → Run all\n")
        print("This happens because Colab needs a restart after installing new packages.")
        raise RuntimeError("Please restart runtime and run all cells again") from e
    else:
        raise

# Test Easy config (2 ambulances, 2 hospitals, 30 steps)
ENV_CONFIG_EASY = {
    "n_ambulances": 2,
    "n_hospitals": 2,
    "max_steps": 30,
    "lambda_param": 0.3,
    "seed": 42,
}

env = AmbulanceEnvironment(ENV_CONFIG_EASY)
obs = env.reset(seed=42)

print(f"✅ Env works!")
print(f"   Ambulances: {len(obs.ambulances)}")
print(f"   Hospitals:  {len(obs.hospitals)}")
print(f"   Step:       {obs.step}")
print(f"   Done:       {obs.done}")

# Test one step with noop
noop = ActionModel(ambulance_id=None, emergency_id="", is_noop=True)
obs2 = env.step(noop)
print(f"   After noop step: reward={obs2.reward:.2f}, done={obs2.done}")

# Test rubric exists
if obs2.rubric:
    print(f"   Rubric total: {obs2.rubric.total():.2f}")
    print(f"   Rubric keys: {list(obs2.rubric.model_dump().keys())}")

In [ ]:
# ============================================================
# CELL 4: Prompt Builder and Rollout Helpers
# Uses exact same format as your train_grpo.py
# ============================================================
import json
import numpy as np
from env.environment import AmbulanceEnvironment
from env.models import ActionModel, AmbulanceState, Severity

SYSTEM_PROMPT = (
    "You are an expert emergency dispatch coordinator for a city ambulance fleet. "
    "Analyze the current situation and dispatch the highest-priority unserved emergency "
    "using the nearest available idle ambulance, routing to the hospital with lowest "
    "occupancy and appropriate specialty. "
    "Your response MUST be valid JSON with exactly these keys: "
    '{"ambulance_id": <int or null>, "emergency_id": "<string>", "hospital_id": <int or null>}. '
    "Set ambulance_id to null and emergency_id to empty string for a no-op. "
    "Priority: CRITICAL > HIGH > NORMAL. "
    "Hospital specialty: CRITICAL→Trauma/Cardiac, HIGH→Trauma/General, NORMAL→General/Paediatric. "
    "Never dispatch to a full hospital (current_patients >= capacity)."
)

def build_prompt(obs):
    """Serialize env observation to LLM-friendly JSON prompt."""
    d = {
        "step": obs.step,
        "ambulances": [
            {
                "id": a.id,
                "node": a.node,
                "state": a.state.value,
                "eta": a.eta,
                "busy": a.state != AmbulanceState.IDLE,
            }
            for a in obs.ambulances
        ],
        "emergencies": [
            {
                "id": e.id,
                "node": e.node,
                "severity": e.severity.value,
                "time_remaining": e.time_remaining,
                "assigned": e.assigned,
            }
            for e in obs.emergencies
            if not e.assigned
        ],
        "hospitals": [
            {
                "id": h.id,
                "node": h.node,
                "occupancy": h.current_patients,
                "capacity": h.capacity,
                "specialty": h.specialty,
                "available": h.current_patients < h.capacity,
            }
            for h in obs.hospitals
        ],
        "traffic_global": obs.traffic.get("global", 1.0) if isinstance(obs.traffic, dict) else 1.0,
    }
    return f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{json.dumps(d, indent=2)}\n<|assistant|>\n"


def parse_response(text, obs):
    """Parse LLM response → ActionModel. Returns noop on any error."""
    try:
        s = text.find("{")
        e = text.rfind("}") + 1
        if s == -1 or e == 0:
            return ActionModel(ambulance_id=None, emergency_id="", is_noop=True)
        p = json.loads(text[s:e])
        idle_ids  = {a.id for a in obs.ambulances if a.state == AmbulanceState.IDLE}
        emg_ids   = {em.id for em in obs.emergencies if not em.assigned}
        hosp_ids  = {h.id for h in obs.hospitals}
        amb_id    = p.get("ambulance_id")
        emg_id    = p.get("emergency_id", "")
        hosp_id   = p.get("hospital_id")
        if (
            amb_id in idle_ids
            and emg_id in emg_ids
            and hosp_id in hosp_ids
        ):
            return ActionModel(
                ambulance_id=amb_id,
                emergency_id=emg_id,
                hospital_id=hosp_id,
            )
    except Exception:
        pass
    return ActionModel(ambulance_id=None, emergency_id="", is_noop=True)


def greedy_baseline(prompt):
    """Rule-based greedy dispatcher: nearest idle → highest priority emergency → least full hospital."""
    try:
        user_part = prompt.split("<|user|>\n")[1].split("\n<|assistant|>")[0]
        obs_json  = json.loads(user_part)
        ambs  = [a for a in obs_json["ambulances"] if not a["busy"]]
        emgs  = sorted(
            obs_json["emergencies"],
            key=lambda e: -{"CRITICAL": 3, "HIGH": 2, "NORMAL": 1}.get(e["severity"], 0)
        )
        hosps = [h for h in obs_json["hospitals"] if h["available"]]
        hosps.sort(key=lambda h: h["occupancy"])
        if ambs and emgs and hosps:
            return json.dumps({
                "ambulance_id": ambs[0]["id"],
                "emergency_id": emgs[0]["id"],
                "hospital_id":  hosps[0]["id"],
            })
    except Exception:
        pass
    return json.dumps({"ambulance_id": None, "emergency_id": "", "hospital_id": None})


def run_episode(generate_fn, env_config=None, seed=None):
    """Run one full episode. Returns (total_reward, rubric_breakdown, n_served, n_missed)."""
    cfg = env_config or ENV_CONFIG_EASY
    e   = AmbulanceEnvironment(cfg)
    o   = e.reset(seed=seed if seed is not None else 42)
    total_r = 0.0
    rubric_acc = {}
    while not o.done:
        p    = build_prompt(o)
        resp = generate_fn(p)
        act  = parse_response(resp, o)
        o    = e.step(act)
        total_r += float(o.reward)
        if o.rubric:
            for k, v in o.rubric.model_dump().items():
                rubric_acc[k] = rubric_acc.get(k, 0.0) + float(v)
    return total_r, rubric_acc, e.metrics.get("served", 0), e.metrics.get("missed", 0)


# Test the functions
_test_env = AmbulanceEnvironment(ENV_CONFIG_EASY)
_test_obs = _test_env.reset(seed=42)
_test_prompt = build_prompt(_test_obs)
print("✅ Prompt builder, parser, and helpers defined.")
print(f"   Greedy test: {greedy_baseline(_test_prompt)[:60]}...")

In [ ]:
# ============================================================
# CELL 5: Load Model
# Uses Unsloth 4-bit if available (fits free T4 15GB VRAM)
# Falls back to standard transformers automatically
# ============================================================
import torch

# Model choice: 0.5B fits T4 free tier comfortably
MODEL_NAME = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"
FALLBACK_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

model, tokenizer = None, None

if USE_UNSLOTH:
    try:
        from unsloth import FastLanguageModel
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=MODEL_NAME,
            max_seq_length=2048,
            load_in_4bit=True,
            dtype=None,  # Auto-detect: bfloat16 on Ampere+, float16 on T4
        )
        model = FastLanguageModel.get_peft_model(
            model,
            r=16,
            target_modules=["q_proj", "v_proj"],
            lora_alpha=16,
            lora_dropout=0,
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=42,
        )
        print(f"✅ Unsloth model loaded: {MODEL_NAME}")
        print(f"   PEFT LoRA r=16 on q_proj, v_proj")
    except Exception as ex:
        print(f"⚠️  Unsloth load failed: {ex}")
        USE_UNSLOTH = False

if not USE_UNSLOTH or model is None:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    print(f"Loading standard transformers model: {FALLBACK_MODEL}")
    tokenizer = AutoTokenizer.from_pretrained(FALLBACK_MODEL, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        FALLBACK_MODEL,
        trust_remote_code=True,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    print(f"✅ Standard model loaded: {FALLBACK_MODEL}")

# Fix pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

device = next(model.parameters()).device
print(f"   Device: {device}")
print(f"   pad_token: {tokenizer.pad_token!r}")
print(f"   Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


In [ ]:
# ============================================================
# CELL 6: Pre-Training Baseline Evaluation
# Records scores BEFORE any GRPO training for the before/after plot
# ============================================================
import torch

def make_generate_fn(mdl, tok, max_new=100, temperature=0.7):
    """Wrap model.generate() into a callable that takes a prompt string."""
    def fn(prompt):
        inputs = tok(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024,
            padding=True,
        ).to(mdl.device)
        with torch.no_grad():
            out = mdl.generate(
                **inputs,
                max_new_tokens=max_new,
                temperature=temperature,
                do_sample=True,
                pad_token_id=tok.pad_token_id,
                eos_token_id=tok.eos_token_id,
            )
        # Return only the newly generated tokens
        new_tokens = out[0][inputs.input_ids.shape[1]:]
        return tok.decode(new_tokens, skip_special_tokens=True)
    return fn

N_EVAL_EPISODES = 8   # 8 seeds for stable mean; takes ~2 min

print(f"📊 Pre-training baseline: evaluating over {N_EVAL_EPISODES} seeds...")
generate_fn = make_generate_fn(model, tokenizer)

greedy_rewards, llm_before_rewards = [], []

for seed in range(N_EVAL_EPISODES):
    # Greedy rule-based
    gr, _, gs, gm = run_episode(greedy_baseline, seed=seed)
    greedy_rewards.append(gr)
    # Untrained LLM
    lr, _, ls, lm = run_episode(generate_fn, seed=seed)
    llm_before_rewards.append(lr)
    print(f"  Seed {seed:2d}: greedy={gr:7.2f}  llm_untrained={lr:7.2f}  "
          f"(greedy served={gs}, llm served={ls})")

print(f"\n📊 Pre-training summary:")
print(f"   Greedy avg:        {np.mean(greedy_rewards):.2f} ± {np.std(greedy_rewards):.2f}")
print(f"   LLM (untrained):   {np.mean(llm_before_rewards):.2f} ± {np.std(llm_before_rewards):.2f}")

# Save for later comparison
import pickle
with open("/content/baseline_scores.pkl", "wb") as f:
    pickle.dump({
        "greedy": greedy_rewards,
        "llm_before": llm_before_rewards,
    }, f)
print("✅ Baseline scores saved to /content/baseline_scores.pkl")

In [ ]:
# ============================================================
# CELL 7: Build Training Dataset
# Creates diverse prompts from many environment seeds and steps
# ============================================================
from datasets import Dataset

TRAINING_STEPS = 100   # Adjust down to 60 if you hit OOM

# Build dataset from diverse seeds — each seed = different city layout
dataset_rows = []
print("Building GRPO training dataset...")
for seed in range(TRAINING_STEPS * 3):
    e = AmbulanceEnvironment(ENV_CONFIG_EASY)
    o = e.reset(seed=seed)
    for step_i in range(4):   # 4 steps per episode for variety
        dataset_rows.append({"prompt": build_prompt(o)})
        # Advance with noop so we get different observation states
        noop = ActionModel(ambulance_id=None, emergency_id="", is_noop=True)
        o = e.step(noop)
        if o.done:
            break

train_dataset = Dataset.from_list(dataset_rows)
print(f"✅ Dataset: {len(train_dataset)} prompts from {TRAINING_STEPS * 3} unique seeds")
print(f"   Sample prompt (first 200 chars): {dataset_rows[0]['prompt'][:200]}...")

In [ ]:
# ============================================================
# CELL 8: Reward Function + GRPOTrainer Configuration
# Reward function directly connects to the real ambulance environment
# ============================================================
from trl import GRPOConfig, GRPOTrainer

def reward_fn(completions, prompts=None, **kwargs):
    """
    Reward function for GRPO.

    For each completion:
    1. Try to parse valid JSON dispatch action
    2. Award +4.0 for valid, non-noop dispatch
    3. Award +1.0 for valid JSON but noop
    4. Penalise -2.0 for malformed JSON
    5. Penalise -5.0 for completely unparseable output

    Bonus: run one real env step and add the environment reward / 10 as signal.
    """
    rewards = []
    for i, completion in enumerate(completions):
        try:
            s = completion.find("{")
            e_idx = completion.rfind("}") + 1
            if s == -1 or e_idx == 0:
                rewards.append(-5.0)
                continue

            parsed = json.loads(completion[s:e_idx])
            has_keys = all(k in parsed for k in ["ambulance_id", "emergency_id", "hospital_id"])

            if not has_keys:
                rewards.append(-2.0)
                continue

            is_noop = (parsed.get("ambulance_id") is None and parsed.get("emergency_id") == "")

            if is_noop:
                # Noop is valid JSON but not useful dispatch — small positive
                rewards.append(0.5)
                continue

            # Valid dispatch: run one env step to get real reward signal
            try:
                env_r = AmbulanceEnvironment(ENV_CONFIG_EASY)
                o_r = env_r.reset(seed=42)
                act = ActionModel(
                    ambulance_id=parsed.get("ambulance_id"),
                    emergency_id=str(parsed.get("emergency_id", "")),
                    hospital_id=parsed.get("hospital_id"),
                )
                step_obs = env_r.step(act)
                env_reward = float(step_obs.reward) / 10.0   # scale down
                base = 4.0 + env_reward
            except Exception:
                base = 3.0   # valid structure but env step failed — still reward JSON

            rewards.append(float(base))

        except (json.JSONDecodeError, ValueError):
            rewards.append(-5.0)

    return rewards


# ──────────────────────────────────────────────────────────────
# GRPO Hyperparameters (tuned for free T4 15GB)
# ──────────────────────────────────────────────────────────────
grpo_config = GRPOConfig(
    output_dir="/content/grpo_output",
    num_train_epochs=1,
    max_steps=TRAINING_STEPS,
    per_device_train_batch_size=1,       # T4 safe
    gradient_accumulation_steps=8,       # effective batch = 8
    learning_rate=5e-6,
    warmup_ratio=0.1,
    logging_steps=5,
    save_steps=50,
    report_to="none",                    # No wandb needed
    num_generations=4,                   # 4 completions per prompt for GRPO
    max_new_tokens=128,
    temperature=0.7,
    bf16=False,
    fp16=True,                           # T4 supports fp16, not bf16
    dataloader_num_workers=0,
    seed=42,
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    config=grpo_config,
    train_dataset=train_dataset,
    reward_funcs=reward_fn,
)

print(f"✅ GRPOTrainer configured.")
print(f"   Training steps:    {TRAINING_STEPS}")
print(f"   Batch size:        {grpo_config.per_device_train_batch_size} × {grpo_config.gradient_accumulation_steps} = {grpo_config.per_device_train_batch_size * grpo_config.gradient_accumulation_steps} effective")
print(f"   Generations/step:  {grpo_config.num_generations}")
print(f"   Learning rate:     {grpo_config.learning_rate}")
print(f"   Max new tokens:    {grpo_config.max_new_tokens}")
print(f"\n⏱️  Estimated time on T4 free tier: {TRAINING_STEPS // 5} – {TRAINING_STEPS // 4} minutes")


In [ ]:
# ============================================================
# CELL 9: RUN GRPO TRAINING
# This is the main training cell. ~15-25 min on T4 GPU.
# Watch the reward numbers — they should trend upward.
# ============================================================
import csv, os

print(f"🚀 Starting GRPO training for {TRAINING_STEPS} steps...")
print("   Watch for reward values — positive trend = LLM learning to dispatch\n")

trainer.train()

# Extract reward log from trainer history
log_history = trainer.state.log_history
reward_log = []
for entry in log_history:
    if "step" in entry:
        r = entry.get("train/reward", entry.get("reward", entry.get("train_reward", None)))
        if r is not None:
            reward_log.append((int(entry["step"]), float(r)))

print(f"\n✅ Training complete!")
print(f"   Log entries with reward: {len(reward_log)}")

if reward_log:
    rewards_only = [r for _, r in reward_log]
    print(f"   First 10 steps avg reward: {np.mean(rewards_only[:10]):.3f}")
    print(f"   Last  10 steps avg reward: {np.mean(rewards_only[-10:]):.3f}")
    delta = np.mean(rewards_only[-10:]) - np.mean(rewards_only[:10])
    print(f"   Improvement (last10 - first10): {delta:+.3f}")

# Save reward CSV
os.makedirs("/content", exist_ok=True)
with open("/content/grpo_rewards.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["step", "reward"])
    writer.writerows(reward_log)
print(f"✅ Reward log saved: /content/grpo_rewards.csv")

# Save trained model
trainer.save_model("/content/grpo_model")
tokenizer.save_pretrained("/content/grpo_model")
print(f"✅ Model saved: /content/grpo_model")


In [ ]:
# ============================================================
# CELL 10: Post-Training Evaluation
# Run same episodes as baseline to get "after" scores
# ============================================================
import pickle

# Load baseline scores
with open("/content/baseline_scores.pkl", "rb") as f:
    baseline_data = pickle.load(f)

greedy_rewards   = baseline_data["greedy"]
llm_before_rewards = baseline_data["llm_before"]

# Evaluate trained model
print(f"📊 Post-training evaluation over {N_EVAL_EPISODES} seeds...")
generate_trained = make_generate_fn(model, tokenizer)
llm_after_rewards = []
llm_after_served  = []

for seed in range(N_EVAL_EPISODES):
    lr, rubric, ls, lm = run_episode(generate_trained, seed=seed)
    llm_after_rewards.append(lr)
    llm_after_served.append(ls)
    print(f"  Seed {seed:2d}: reward={lr:7.2f}  served={ls}")

print(f"\n📊 Post-training summary:")
print(f"   Greedy avg:        {np.mean(greedy_rewards):.2f}")
print(f"   LLM before GRPO:   {np.mean(llm_before_rewards):.2f}")
print(f"   LLM after GRPO:    {np.mean(llm_after_rewards):.2f}")
print(f"   Δ improvement:     {np.mean(llm_after_rewards) - np.mean(llm_before_rewards):+.2f}")


In [ ]:
# ============================================================
# CELL 11: Generate Plots
# Creates 3 plots that go directly into your README and blog post
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

plt.style.use("dark_background")
COLORS = {
    "greedy":    "#6b7280",
    "before":    "#f59e0b",
    "after":     "#10b981",
    "curve":     "#6366f1",
    "ma":        "#a5b4fc",
}

# ─── PLOT 1: GRPO Training Reward Curve ───────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(10, 5))

if reward_log:
    steps  = [s for s, _ in reward_log]
    values = [r for _, r in reward_log]
    window = min(10, len(values) // 3 or 1)
    ma     = np.convolve(values, np.ones(window) / window, mode="valid")

    ax1.plot(steps, values, alpha=0.3, color=COLORS["curve"], linewidth=1.0, label="Per-step reward")
    ax1.plot(steps[window-1:], ma, color=COLORS["ma"], linewidth=2.5,
             label=f"{window}-step moving average")
    ax1.axhline(0, color="#4b5563", linestyle="--", linewidth=0.8)

    # Annotate trend
    if len(values) >= 20:
        first_avg = np.mean(values[:10])
        last_avg  = np.mean(values[-10:])
        ax1.annotate(f"Start: {first_avg:.2f}", xy=(steps[5], first_avg),
                     xytext=(steps[5], first_avg + 0.5),
                     color="#9ca3af", fontsize=9, ha="center")
        ax1.annotate(f"End: {last_avg:.2f}", xy=(steps[-5], last_avg),
                     xytext=(steps[-5], last_avg + 0.5),
                     color=COLORS["after"], fontsize=9, ha="center")

ax1.set_title("GRPO Training Reward Curve — Ambulance Dispatch LLM", fontsize=14, pad=12)
ax1.set_xlabel("Training Step", fontsize=12)
ax1.set_ylabel("Reward", fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(alpha=0.2)
plt.tight_layout()
plt.savefig("/content/grpo_reward_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: /content/grpo_reward_curve.png")

# ─── PLOT 2: Before/After Bar Chart ───────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(10, 5))

x = np.arange(N_EVAL_EPISODES)
w = 0.28

bars_g = ax2.bar(x - w,     greedy_rewards,     w, label="Greedy Baseline",      color=COLORS["greedy"],  alpha=0.85)
bars_b = ax2.bar(x,          llm_before_rewards, w, label="LLM — Untrained",      color=COLORS["before"],  alpha=0.85)
bars_a = ax2.bar(x + w,      llm_after_rewards,  w, label="LLM — GRPO Trained",   color=COLORS["after"],   alpha=0.85)

ax2.axhline(np.mean(greedy_rewards),     color=COLORS["greedy"], linestyle="--", linewidth=1.2, alpha=0.6)
ax2.axhline(np.mean(llm_before_rewards), color=COLORS["before"], linestyle="--", linewidth=1.2, alpha=0.6)
ax2.axhline(np.mean(llm_after_rewards),  color=COLORS["after"],  linestyle="--", linewidth=1.8,
            label=f"GRPO avg = {np.mean(llm_after_rewards):.1f}")

ax2.set_xticks(x)
ax2.set_xticklabels([f"Ep {i+1}" for i in range(N_EVAL_EPISODES)], fontsize=9)
ax2.set_title("Episode Reward: Baseline vs GRPO-Trained LLM (Ambulance Dispatch)", fontsize=13, pad=12)
ax2.set_xlabel("Evaluation Episode (seed 0–7)", fontsize=11)
ax2.set_ylabel("Total Episode Reward", fontsize=11)
ax2.legend(fontsize=10)
ax2.grid(alpha=0.15, axis="y")
plt.tight_layout()
plt.savefig("/content/grpo_before_after.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: /content/grpo_before_after.png")

# ─── PLOT 3: RFC 004 Rubric Breakdown ─────────────────────────────────────────
print("\n📊 Running rubric breakdown evaluation (3 episodes each)...")
rubric_greedy, rubric_trained = {}, {}

for seed in range(3):
    _, rg, _, _ = run_episode(greedy_baseline, seed=seed)
    _, ra, _, _ = run_episode(generate_trained, seed=seed)
    for k, v in rg.items():
        rubric_greedy[k] = rubric_greedy.get(k, 0.0) + v / 3.0
    for k, v in ra.items():
        rubric_trained[k] = rubric_trained.get(k, 0.0) + v / 3.0

RUBRIC_KEYS = [
    "emergency_served", "severity_bonus", "dispatch_speed",
    "hospital_delivery", "distance_penalty", "traffic_penalty",
    "idle_penalty", "capacity_violation", "timeout_penalty",
]
labels     = [k.replace("_", "\n") for k in RUBRIC_KEYS]
vals_g     = [rubric_greedy.get(k, 0.0) for k in RUBRIC_KEYS]
vals_t     = [rubric_trained.get(k, 0.0) for k in RUBRIC_KEYS]

fig3, ax3 = plt.subplots(figsize=(14, 6))
x3 = np.arange(len(RUBRIC_KEYS))
w3 = 0.38
ax3.bar(x3 - w3/2, vals_g, w3, label="Greedy Baseline", color=COLORS["greedy"], alpha=0.85)
ax3.bar(x3 + w3/2, vals_t, w3, label="GRPO Trained",    color=COLORS["after"],  alpha=0.85)
ax3.axhline(0, color="#4b5563", linewidth=0.8)
ax3.set_xticks(x3)
ax3.set_xticklabels(labels, fontsize=8)
ax3.set_title("RFC 004 Rubric Breakdown — 9 Reward Components (avg over 3 episodes)", fontsize=13, pad=12)
ax3.set_xlabel("Rubric Component", fontsize=11)
ax3.set_ylabel("Cumulative Component Reward", fontsize=11)
ax3.legend(fontsize=10)
ax3.grid(alpha=0.15, axis="y")
plt.tight_layout()
plt.savefig("/content/rubric_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: /content/rubric_breakdown.png")

print("\n✅ All 3 plots generated!")
print("   /content/grpo_reward_curve.png")
print("   /content/grpo_before_after.png")
print("   /content/rubric_breakdown.png")


In [ ]:
# ============================================================
# CELL 12: Print Summary Numbers
# Copy these exact numbers into your README results table and blog post
# ============================================================
print("=" * 60)
print("  COPY THESE NUMBERS INTO README AND BLOG POST")
print("=" * 60)
print(f"\n  Greedy baseline avg reward:   {np.mean(greedy_rewards):.2f}")
print(f"  LLM untrained avg reward:     {np.mean(llm_before_rewards):.2f}")
print(f"  LLM GRPO-trained avg reward:  {np.mean(llm_after_rewards):.2f}")
print(f"  Delta (trained - untrained):  {np.mean(llm_after_rewards) - np.mean(llm_before_rewards):+.2f}")
print(f"\n  GRPO training steps:          {TRAINING_STEPS}")
print(f"  Model:                        Qwen2.5-0.5B-Instruct (4-bit via Unsloth)")

if reward_log:
    rv = [r for _, r in reward_log]
    print(f"  Reward at step 1-10:          {np.mean(rv[:10]):.3f}")
    print(f"  Reward at step final 10:      {np.mean(rv[-10:]):.3f}")
    print(f"  Best step reward:             {max(rv):.3f}")
print("=" * 60)


In [ ]:
# ============================================================
# CELL 13: Download All Files
# Run this and download EACH file when the download dialog appears
# ============================================================
from google.colab import files
import time

files_to_download = [
    "/content/grpo_reward_curve.png",
    "/content/grpo_before_after.png",
    "/content/rubric_breakdown.png",
    "/content/grpo_rewards.csv",
]

for f_path in files_to_download:
    import os
    if os.path.exists(f_path):
        print(f"📥 Downloading {f_path}...")
        files.download(f_path)
        time.sleep(1)  # Brief pause between downloads
    else:
        print(f"⚠️  File not found: {f_path} — check if plot cells ran successfully")

print("\n✅ Download complete!")
print("   Save all 4 files to: C:\\Users\\visha\\Downloads\\Ambulance-OpenENV\\")
print("   These exact filenames are already referenced in your README.md")
